# Explainable AI-Based Air Pollution Prediction and Source Attribution

**Repository:** https://github.com/BhanuDanda-09/Air-Pollution-XAI.git  
**Phases Covered:**
- **Phase 1:** Dataset Inspection, Leak-Free Preprocessing, Exploratory Data Analysis, and Baseline Random Forest Regressor
- **Phase 2:** XGBoost Regressor Development, 5-Fold Cross-Validated Hyperparameter Tuning, and Cross-Model Comparison

---
### Notebook Objectives:
1. **Dataset Inspection:** Audit raw dataset `Delhi Final(1).xlsx` (dimensions, data types, missingness, duplicates, physical domains).
2. **Data Preprocessing:** Remove empty sheet artifacts, eliminate 156 duplicate rows to prevent train-test data leakage, build a median imputer pipeline, and split into 80% train / 20% test partitions (`random_state=42`).
3. **Exploratory Data Analysis (EDA):** Generate 9 publication-grade diagnostic visualizations.
4. **Baseline Modeling (Random Forest):** Train a `RandomForestRegressor` pipeline and evaluate on held-out test data.
5. **Phase 2 Modeling (XGBoost):** Optimize an `XGBRegressor` using 5-fold cross-validation (`RandomizedSearchCV`) on training data.
6. **Cross-Model Evaluation:** Compare Random Forest and XGBoost head-to-head on the exact same 406 test samples.
7. **Inference Verification:** Validate prediction on a live unseen sample using both models.


In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styling
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.dpi'] = 150
print('Libraries imported successfully.')


## 1. Dataset Loading & Inspection
We load the raw Excel dataset `Delhi Final(1).xlsx` and inspect its properties.


In [ ]:
dataset_path = '../dataset/Delhi Final(1).xlsx'
raw_df = pd.read_excel(dataset_path)

print('Raw Dataset Dimensions:', raw_df.shape)
print('\nAll Detected Columns:', raw_df.columns.tolist())
raw_df.head()


In [ ]:
# Inspect the 6 core variables
core_cols = ['PM2.5', 'RH', 'WD', 'SR', 'BP', 'AT']
core_df = raw_df[core_cols]

print('--- Missing Values in Core Variables ---')
print(core_df.isnull().sum())

print('\n--- Statistical Summary ---')
display(core_df.describe().T)

print('\nExact duplicate rows in core features:', core_df.duplicated().sum())


## 2. Preprocessing & Leakage-Free Train-Test Split

1. **Artifact Removal:** Dropped empty artifact columns `Unnamed: 6` to `Unnamed: 12`.
2. **Deduplication:** Dropped 156 duplicate rows to ensure cross-fold independence and eliminate train-test leakage.
3. **Train-Test Split:** 80% Training (1,621 samples) and 20% Testing (406 samples) with `random_state=42`.
4. **Imputation:** `SimpleImputer(strategy='median')` fitted exclusively on `X_train`.


In [ ]:
# Deduplicate
cleaned_df = core_df.drop_duplicates().reset_index(drop=True)
print('Cleaned Dataset Shape:', cleaned_df.shape)

# Separate Features (X) and Target (y)
features = ['RH', 'WD', 'SR', 'BP', 'AT']
target = 'PM2.5'

X = cleaned_df[features]
y = cleaned_df[target]

# Train-test split (80% train, 20% test, seed=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

print(f'Training Set: {X_train.shape[0]} samples')
print(f'Testing Set:  {X_test.shape[0]} samples')


## 3. Exploratory Data Analysis (EDA) Visualizations


In [ ]:
# Target distribution and boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(cleaned_df['PM2.5'], kde=True, color='#d95f02', bins=35, ax=axes[0], edgecolor='k')
pm_mean = cleaned_df['PM2.5'].mean()
pm_median = cleaned_df['PM2.5'].median()
axes[0].axvline(pm_mean, color='red', linestyle='--', label=f'Mean: {pm_mean:.1f}')
axes[0].axvline(pm_median, color='blue', linestyle='-', label=f'Median: {pm_median:.1f}')
axes[0].set_title('PM2.5 Concentration Distribution', fontweight='bold')
axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].legend()

sns.boxplot(x=cleaned_df['PM2.5'], color='#fc8d62', ax=axes[1])
axes[1].set_title('PM2.5 Concentration Boxplot', fontweight='bold')
axes[1].set_xlabel('PM2.5 (µg/m³)')

plt.tight_layout()
plt.show()


In [ ]:
# Pearson correlation heatmap
plt.figure(figsize=(8, 6))
corr = cleaned_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=1.0)
plt.title('Pearson Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots of features vs PM2.5
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

scatter_configs = [
    ('RH', 'Relative Humidity (%)', '#1f78b4'),
    ('WD', 'Wind Direction (°)', '#33a02c'),
    ('SR', 'Solar Radiation (W/m²)', '#e31a1c'),
    ('BP', 'Barometric Pressure (mmHg)', '#ff7f00'),
    ('AT', 'Air Temperature (°C)', '#6a3d9a'),
]

for idx, (col, lbl, clr) in enumerate(scatter_configs):
    ax = axes[idx]
    sns.regplot(data=cleaned_df, x=col, y='PM2.5', scatter_kws={'alpha': 0.35, 'color': clr, 's': 20},
                line_kws={'color': 'black', 'linewidth': 1.8}, ax=ax)
    r_val = corr.loc[col, 'PM2.5']
    ax.set_title(f'{lbl} vs PM2.5 (r = {r_val:+.2f})', fontweight='bold')
    ax.set_xlabel(lbl)
    ax.set_ylabel('PM2.5 (µg/m³)')

fig.delaxes(axes[5])
plt.suptitle('Meteorological Variables vs. PM2.5 Concentration', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 4. Phase 1: Baseline Random Forest Regressor


In [ ]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
joblib.dump(rf_pipeline, '../models/random_forest.pkl')

y_pred_rf = rf_pipeline.predict(X_test)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print(f'Random Forest Test MAE:  {rf_mae:.2f} µg/m³')
print(f'Random Forest Test RMSE: {rf_rmse:.2f} µg/m³')
print(f'Random Forest Test R²:   {rf_r2:.4f} ({rf_r2*100:.2f}%)')


## 5. Phase 2: XGBoost Regressor & 5-Fold Cross-Validation Tuning

We optimize an `XGBRegressor` pipeline using `RandomizedSearchCV` on `X_train` with 5-fold cross-validation.


In [ ]:
xgb_base = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', XGBRegressor(random_state=42, n_jobs=2, objective='reg:squarederror', tree_method='hist'))
])

param_grid = {
    'regressor__n_estimators': [50, 100, 150, 200, 250],
    'regressor__max_depth': [3, 4, 5, 6, 7],
    'regressor__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'regressor__subsample': [0.7, 0.8, 0.9, 1.0],
    'regressor__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'regressor__min_child_weight': [1, 2, 3, 5],
    'regressor__reg_lambda': [1.0, 2.0, 5.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=0
)

print('Tuning XGBoost hyperparameters with 5-fold CV...')
search.fit(X_train, y_train)

best_xgb = search.best_estimator_
print('Best 5-Fold CV Training RMSE:', -search.best_score_)
print('Best Hyperparameters:', search.best_params_)

# Serialize
joblib.dump(best_xgb, '../models/xgboost_model.pkl')

y_pred_xgb = best_xgb.predict(X_test)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
xgb_r2 = r2_score(y_test, y_pred_xgb)

print(f'\nXGBoost Test MAE:  {xgb_mae:.2f} µg/m³')
print(f'XGBoost Test RMSE: {xgb_rmse:.2f} µg/m³')
print(f'XGBoost Test R²:   {xgb_r2:.4f} ({xgb_r2*100:.2f}%)')


## 6. Head-to-Head Model Comparison (Random Forest vs. XGBoost)

We compare both models on the exact same 406 held-out test samples.


In [ ]:
comp_table = pd.DataFrame([
    {
        'Model': 'Random Forest (Baseline)',
        'MAE (µg/m³)': f'{rf_mae:.2f}',
        'MSE (µg/m²)²': f'{mean_squared_error(y_test, y_pred_rf):.2f}',
        'RMSE (µg/m³)': f'{rf_rmse:.2f}',
        'R² Score': f'{rf_r2:.4f}'
    },
    {
        'Model': 'XGBoost (Tuned)',
        'MAE (µg/m³)': f'{xgb_mae:.2f}',
        'MSE (µg/m²)²': f'{mean_squared_error(y_test, y_pred_xgb):.2f}',
        'RMSE (µg/m³)': f'{xgb_rmse:.2f}',
        'R² Score': f'{xgb_r2:.4f}'
    }
])
display(comp_table)


In [ ]:
# Comparative visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models = ['Random Forest', 'XGBoost']
colors = ['#1f77b4', '#e6550d']

# MAE
axes[0].bar(models, [rf_mae, xgb_mae], color=colors, width=0.45, edgecolor='k')
for i, v in enumerate([rf_mae, xgb_mae]):
    axes[0].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')
axes[0].set_title('Mean Absolute Error (MAE)', fontweight='bold')
axes[0].set_ylabel('µg/m³ [Lower is Better]')
axes[0].set_ylim(0, max(rf_mae, xgb_mae) * 1.25)

# RMSE
axes[1].bar(models, [rf_rmse, xgb_rmse], color=colors, width=0.45, edgecolor='k')
for i, v in enumerate([rf_rmse, xgb_rmse]):
    axes[1].text(i, v + 0.8, f'{v:.2f}', ha='center', fontweight='bold')
axes[1].set_title('Root Mean Squared Error (RMSE)', fontweight='bold')
axes[1].set_ylabel('µg/m³ [Lower is Better]')
axes[1].set_ylim(0, max(rf_rmse, xgb_rmse) * 1.25)

# R2
axes[2].bar(models, [rf_r2, xgb_r2], color=colors, width=0.45, edgecolor='k')
for i, v in enumerate([rf_r2, xgb_r2]):
    axes[2].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
axes[2].set_title('R² Score (Variance Explained)', fontweight='bold')
axes[2].set_ylabel('Dimensionless [Higher is Better]')
axes[2].set_ylim(0, 1.0)

plt.suptitle('Random Forest vs. XGBoost: Test Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Comparative Feature Importance Plot
rf_imp = rf_pipeline.named_steps['regressor'].feature_importances_
xgb_imp = best_xgb.named_steps['regressor'].feature_importances_

x_pos = np.arange(len(features))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x_pos - width/2, rf_imp * 100, width, label='Random Forest (Gini Importance)', color='#1f77b4', edgecolor='k')
plt.bar(x_pos + width/2, xgb_imp * 100, width, label='XGBoost (Gain Importance)', color='#e6550d', edgecolor='k')
plt.xticks(x_pos, features, fontsize=11)
plt.ylabel('Relative Feature Contribution (%)', fontsize=11)
plt.title('Feature Importance Comparison: Random Forest vs. XGBoost', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()


## 7. Live Inference Verification on Unseen Test Sample


In [ ]:
sample_x = X_test.iloc[[0]]
actual_pm25 = y_test.iloc[0]

pred_rf = rf_pipeline.predict(sample_x)[0]
pred_xgb = best_xgb.predict(sample_x)[0]

print('Sample Input Features:')
display(sample_x)
print(f'Ground Truth PM2.5:     {actual_pm25:.2f} µg/m³')
print(f'Random Forest Forecast: {pred_rf:.2f} µg/m³ (Error: {abs(actual_pm25 - pred_rf):.2f} µg/m³)')
print(f'XGBoost Forecast:       {pred_xgb:.2f} µg/m³ (Error: {abs(actual_pm25 - pred_xgb):.2f} µg/m³)')


## 8. Scientific Limitations & Future Scope

1. **No Timestamp Column:** The dataset does not contain temporal headers. It cannot support genuine autoregressive time-series forecasting without timestamps.
2. **Feature Boundaries:** The dataset contains only PM2.5 and 5 meteorological variables (`RH`, `WD`, `SR`, `BP`, `AT`). No additional pollutant columns were fabricated.
3. **Feature Importance vs. Source Attribution:** Tree feature importance measures model variance/loss reduction. It does not prove chemical emission causation or distinguish physical pollution sources.
4. **Next Phase (Phase 3):** Explainable AI integration with SHAP (TreeExplainer, summary beeswarm plots, dependence plots, waterfall plots) and LIME.
